Thư viện

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split # Chia dữ liệu thành tập huấn luyện và tập kiểm tra
from sklearn.preprocessing import StandardScaler # Chuẩn hóa dữ liệu
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, recall_score
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.model_selection import GridSearchCV
import pickle

Đọc file đặc trưng

In [3]:
data = pd.read_csv('features.csv')

Tách features(x) và label(y)

In [5]:
X = data.drop('label', axis=1) # axis 1 để chỉ cột
y = data['label'] # Lấy cột label

Chia dữ liệu thành tập huấn luyện và tập kiểm tra

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # 20% dữ liệu dùng để kiểm tra

Chuẩn hóa dữ liệu

In [10]:
scaler = StandardScaler()
X_train_scale = scaler.fit_transform(X_train) # Chuẩn hóa dữ liệu huấn luyện
X_test_scale = scaler.transform(X_test) # Chuẩn hóa dữ liệu kiểm tra


 Bộ tham số để GridSearchCV điều chỉnh

In [7]:
parameters = {"n_estimators": [100, 200, 300],
              "max_depth": [None, 10, 20, 30],
              "min_samples_split": [2, 5, 10],
            }

Tạo mô hình với GridSearchCV để tìm tham số tốt nhất

In [ ]:
Model = GridSearchCV(RandomForestClassifier(), parameters, cv=5) 

Huấn luyện mô hình

In [ ]:
Model.fit(X_train_scale, y_train) # Huấn luyện mô hình

Lấy mô hình tốt nhất từ GridSearchCV

In [ ]:
Best_model= Model.best_estimator_ 

Dự đoán

In [ ]:
train_predict = Model.predict(X_train_scale)
test_predict = Model.predict(X_test_scale)

Đánh giá mô hình

In [ ]:
# Macro F1-score
macro_f1 = f1_score(y_test, test_predict, average='macro')

# Accuracy
train_accuracy = accuracy_score(y_train, train_predict)
test_accuracy = accuracy_score(y_test, test_predict)

#Perclass Recall
perclass_recall = recall_score(y_test, test_predict, average=None)
perclass_recall = [round(r, 4) for r in perclass_recall]

# Confusion Matrix
train_cm = confusion_matrix(y_train, train_predict)
test_cm = confusion_matrix(y_test, test_predict)




vẽ Confusion Matrix kèm Accuracy và F1-score

In [ ]:
Fault_types = [ 'BPFI_03',
                'BPFI_10', 
                'BPFO_03', 
                'BPFO_10', 
                'Misalign_01', 
                'Misalign_03',
                'Unbalance_0583mg',
                'Unbalance_1169mg',
                'Normal',
                'Normal_2Nm',
                'Normal_4Nm',
                ]      
#Biểu đồ confusion matrix
plt.figure(figsize=(12, 6))# Kích thước biểu đồ
plt.subplot(1, 2, 1)# Vẽ biểu đồ đầu tiên trong 1 hàng 2 cột
sns.heatmap(train_cm, annot=True, fmt='d', cmap='Blues', 
xticklabels=Fault_types, yticklabels=Fault_types)# Vẽ heatmap với các nhãn trục x và y
plt.title(f'Training Confusion Matrix\n\nAccuracy: {train_accuracy:.4f}' )# Tiêu đề biểu đồ
plt.xlabel('Predicted')# Nhãn trục x
plt.ylabel('Actual')# Nhãn trục y
plt.subplot(1, 2, 2)# Vẽ biểu đồ thứ hai trong 1 hàng 2 cột
sns.heatmap(test_cm, annot=True, fmt='d', cmap='Blues',
xticklabels=Fault_types, yticklabels=Fault_types)# Vẽ heatmap với các nhãn trục x và y
plt.title(f'Testing Confusion Matrix\n\n Macro-F1: {macro_f1:.4f}\n\nAccuracy: {test_accuracy:.4f}')# Tiêu đề biểu đồ
plt.xlabel('Predicted')# Nhãn trục x
plt.ylabel('Actual')# Nhãn trục y
plt.tight_layout()# Tự động điều chỉnh khoảng cách giữa các phần tử trong biểu đồ
plt.show()# Hiển thị biểu đồ


Bảng Perclass Recall

In [ ]:
df_recall = pd.DataFrame({
    "STT": range(1, len(Fault_types) + 1),
    "Falut": Fault_types,
    "Recall": perclass_recall
})
# Vẽ bảng 
fig, ax = plt.subplots(figsize=(6, 3))  
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=df_recall.values, colLabels=df_recall.columns, loc='center')

plt.show()


Tạo đối tượng giải thích mô hình RF

In [ ]:
Explainer = shap.Explainer(Best_model, X_train_scale) # Tạo đối tượng giải thích mô hình
shap_values = Explainer(X_test_scale) # Tính giá trị SHAP cho tập kiểm tra

Vẽ biểu đồ thanh SHAP

In [ ]:
plt.figure()
shap.bar_plot(shap_values, max_display=10) # Biểu đồ thanh giá trị SHAP
plt.show()